In [1]:
import pandas as pd
import numpy as np

---

## Level 1 — Retail inventory

Quick reminder — MultiIndex:
```python
df = df.set_index(['a', 'b'])           # set MultiIndex
df.loc[('NYC', 'Laptop')]               # retrieve a specific entry
df.loc['NYC']                           # all rows for NYC
```

- Set `['store', 'product']` as the index.
- Retrieve the NYC Laptop row. Then retrieve all LA rows.
- Add a `value` column (`units × price`). Which store has the highest total inventory value? Use `np.sum` per group.
- Which store × product combination has the most units? Use `np.argmax`.

In [13]:
inventory = pd.DataFrame({
    'store':    ['NYC','NYC','NYC','LA','LA','LA','CHI','CHI','CHI'],
    'product':  ['Laptop','Phone','Tablet','Laptop','Phone','Tablet','Laptop','Phone','Tablet'],
    'units':    [15, 42, 28, 8, 35, 19, 22, 51, 31],
    'price':    [999, 699, 449, 999, 699, 449, 999, 699, 449],
    'restocked':['2024-01-15','2024-01-20','2024-02-01','2024-01-18','2024-01-25',
                 '2024-02-05','2024-01-10','2024-01-22','2024-02-08'],
})

# Your code here

inventory = inventory.set_index(['store','product'])

print(inventory.loc['NYC','Laptop'])
print(inventory.loc['LA'])

inventory['value'] = inventory['units']*inventory['price']
print(inventory.groupby(level = 'store')['value'].sum().idxmax(),'has the highest total value')

print(inventory.index[np.argmax(inventory['units'])],'has the most units')

units                15
price               999
restocked    2024-01-15
Name: (NYC, Laptop), dtype: object
         units  price   restocked
product                          
Laptop       8    999  2024-01-18
Phone       35    699  2024-01-25
Tablet      19    449  2024-02-05
CHI has the highest total value
('CHI', 'Phone') has the most units


---

## Level 2 — Employee satisfaction

Quick reminder — `category` dtype:
```python
df['col'] = df['col'].astype('category')                     # basic conversion
df.memory_usage(deep=True).sum()                             # check memory

# Ordered categories enable < > comparisons:
order = pd.CategoricalDtype(['Low','Medium','High'], ordered=True)
df['col'] = df['col'].astype(order)
df[df['col'] > 'Low']                                        # works with ordered
```

1. Record memory usage before any conversion.
2. Convert `dept`, `level`, and `satisfaction` to `category`. How much memory did you save?
3. Make `satisfaction` an ordered category (`Low < Medium < High`). Filter to rows where satisfaction is above Low.
4. Which department has the most High-satisfaction responses?
5. What are the mean and 75th percentile scores across all employees? Use `np.mean` and `np.percentile`.

In [27]:
responses = pd.DataFrame({
    'emp_id':       range(1, 21),
    'dept':         ['Engineering']*7 + ['Marketing']*6 + ['Sales']*7,
    'level':        ['Junior','Senior','Lead','Junior','Senior','Junior','Lead',
                     'Junior','Senior','Junior','Senior','Junior','Senior',
                     'Junior','Junior','Senior','Lead','Junior','Senior','Lead'],
    'satisfaction': ['Low','High','Medium','High','High','Medium','High',
                     'Low','Medium','High','Low','Medium','High',
                     'Medium','Low','High','High','Medium','Low','High'],
    'score':        [45, 88, 72, 85, 91, 68, 93,
                     42, 65, 81, 38, 70, 87,
                     67, 44, 90, 88, 73, 41, 86],
})

# Your code here

bm = responses.memory_usage(deep = True).sum()
print(bm)

responses[['dept','level','satisfaction']] = responses[['dept','level','satisfaction']].astype('category')

print(bm - responses.memory_usage(deep = True).sum(),'saved')

so = pd.CategoricalDtype(['Low','Medium','High'], ordered=True)
responses['satisfaction'] = responses['satisfaction'].astype(so)
f = responses.loc[responses['satisfaction']>'Low']

gs = responses.groupby('dept', observed=True)['satisfaction'].apply(lambda x: (x=='High').sum())
print(gs.idxmax(),'has the most high satification')


print('mean is', np.mean(responses['score']))

print('75th percentile is', np.percentile(responses['score'],75))

4233
2834 saved
Engineering has the most high satification
mean is 70.7
75th percentile is 87.25


---

## Level 3 — Trade execution analysis

Quick reminder — `merge_asof`:
```python
# Both DataFrames must be sorted by the key column first
pd.merge_asof(left, right, on='time')
# Each left row is matched to the most recent right row where right.time <= left.time
```

A stock exchange recorded trades and quote updates separately. Match each trade to the market quote that was active at the moment of the trade.

No steps:

1. Use `pd.merge_asof()` to join each trade to its most recent quote.
2. Add a `spread` column (`ask - bid`). What was the average spread across all trades?
3. Which trade had the widest spread at execution?
4. Was there a correlation between trade size and spread? Use `np.corrcoef`.
5. Filter to trades where the trade price exceeded the ask — these indicate aggressive buying. How many were there?

In [36]:
trades = pd.DataFrame({
    'time':  pd.to_datetime(['2024-01-02 09:30:00','2024-01-02 09:31:15',
                             '2024-01-02 09:33:00','2024-01-02 09:34:30',
                             '2024-01-02 09:36:00','2024-01-02 09:37:45',
                             '2024-01-02 09:39:00','2024-01-02 09:40:30']),
    'price': [150.50, 151.20, 150.80, 152.10, 151.90, 153.40, 152.80, 154.20],
    'size':  [100, 250, 150, 300, 200, 175, 225, 400],
})

quotes = pd.DataFrame({
    'time': pd.to_datetime(['2024-01-02 09:29:45','2024-01-02 09:30:30',
                            '2024-01-02 09:32:00','2024-01-02 09:33:45',
                            '2024-01-02 09:35:15','2024-01-02 09:37:00',
                            '2024-01-02 09:38:30','2024-01-02 09:40:00']),
    'bid':  [150.25, 150.75, 151.00, 150.50, 151.75, 153.00, 152.50, 153.90],
    'ask':  [150.50, 151.00, 151.25, 150.75, 152.00, 153.25, 152.75, 154.15],
})

# Your code here
trades = trades.sort_values('time')
quotes = quotes.sort_values('time')

m = pd.merge_asof(trades, quotes, on = 'time')
m['spread'] = m['ask'] - m['bid']
print('average is', m['spread'].mean())
print(m.loc[m['spread'].idxmax()],'has the widest spread')

print('spread are all the same')

f = m.loc[m['price']>m['ask']]
print(f.shape[0],'has price exceeded ask')

average is 0.25
time      2024-01-02 09:30:00
price                   150.5
size                      100
bid                    150.25
ask                     150.5
spread                   0.25
Name: 0, dtype: object has the widest spread
spread are all the same
5 has price exceeded ask
